# Phylogenetic Tree Recovery — Full Diagnostic Run

End-to-end runner for `rep-phylogeny`'s diagnostic suite (`diagnostics.md`).

**What it does**
1. Extracts EVERY transformer layer for `xlm-roberta-large` (24, fp16, mean-pool)
   and `google/gemma-2-2b` (26, fp32, mean-pool by default; add `last_token`
   via `--pools mean_pool last_token` if you have the time budget).
2. Runs Blocks A–H of the diagnostic suite for each `(model, pool, layer) ×
   (procrustes, ridge)` configuration.
3. Writes per-config logs and a single `SUMMARY.txt`.

**Setup**
- Accelerator: **GPU T4 x2**.
- Add a Kaggle Secret named `HF_TOKEN` and accept the Gemma-2 license.
- Default cell-6 command uses Gemma `mean_pool` only and `--n-perms 10`.
  Expected wall time: ≈45 min extraction + ≈3 h diagnostics ≈ **3.5–4 h** total.

## 1. Clone repo and install deps

In [ ]:
import os, subprocess
WORKDIR = "/kaggle/working"
REPO_DIR = os.path.join(WORKDIR, "rep-phylogeny")
os.chdir(WORKDIR)
if os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--rebase"])
else:
    subprocess.check_call(["git", "clone", "https://github.com/saketatreya/rep-phylogeny.git", REPO_DIR])
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!pip install -q -r requirements.txt

## 2. HuggingFace token (required for Gemma-2-2b)

Add a Kaggle Secret named `HF_TOKEN` (Add-ons → Secrets) and accept the
[Gemma-2 license](https://huggingface.co/google/gemma-2-2b). Without this,
run only XLM-R by passing `--models xlm-roberta-large` to the cell below.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
    print("HF token loaded.")
except Exception as e:
    print(f"No HF token: {e}")
    print("Gemma-2 will fail. Run only XLM-R below by passing --models xlm-roberta-large.")

## 3. Run full diagnostics

Both models, all layers, both methods (Procrustes + Ridge), all
Blocks A–H. Gemma uses `mean_pool` only and `n_perms=10` so the whole run
fits in ~3.5–4 h on Kaggle's T4×2.

Outputs land in `/kaggle/working/results_diag/`. To also sweep Gemma's
`last_token` pool (adds ~2.5 h), append `--pools mean_pool last_token` and
remove `--pools mean_pool` from the command.

In [ ]:
!python run_diagnostics.py \
    --out-dir /kaggle/working/results_diag \
    --pools mean_pool \
    --n-perms 10

## 4. Inspect the master summary

In [ ]:
print(open("/kaggle/working/results_diag/SUMMARY.txt").read())

## 5. (Optional) drill into a single config

Per-config logs live at `results_diag/{model}/{pool}/{layer}/{method}.log`
and `block_a.log`. The `sanity.log` at `results_diag/{model}/{pool}/` lists
norms and Spa-Por/Fre/Ron cosines for every layer.

In [ ]:
# Example: XLM-R, mean_pool, layer_12, ridge
p = "/kaggle/working/results_diag/xlm-roberta-large/mean_pool/layer_12/ridge.log"
print(open(p).read())